# Testing the Underdamped Langevin Sampler

This notebook tests the underdamped Langevin sampler in Blackjax, focusing on:
1. Verifying the mass matrix application
2. Confirming the energy change computation
3. Visualizing the sampler's behavior

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax import random

from blackjax.mcmc.integrators import velocity_verlet
from blackjax.mcmc.metrics import default_metric

# Import Blackjax components
from blackjax.mcmc.underdamped_langevin import as_top_level_api, build_kernel, init
from blackjax.types import PRNGKey

# Set random seed for reproducibility
key = random.PRNGKey(0)

## 1. Define a Simple Target Distribution

Let's start with a simple 2D Gaussian distribution to test the sampler.

In [ ]:
def logdensity_fn(position):
    """Log-density of a 2D Gaussian distribution."""
    # Mean and covariance matrix
    mean = jnp.array([0.0, 0.0])
    cov = jnp.array([[1.0, 0.5], [0.5, 2.0]])

    # Compute log-density
    diff = position - mean
    return -0.5 * jnp.dot(diff, jnp.linalg.solve(cov, diff))


# Visualize the target distribution
def plot_target_distribution():
    x = np.linspace(-4, 4, 100)
    y = np.linspace(-4, 4, 100)
    X, Y = np.meshgrid(x, y)

    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = np.exp(logdensity_fn(jnp.array([X[i, j], Y[i, j]])))

    plt.figure(figsize=(10, 8))
    plt.contourf(X, Y, Z, levels=20, cmap="viridis")
    plt.colorbar(label="Density")
    plt.title("Target Distribution")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()


plot_target_distribution()

## 2. Test Mass Matrix Application

Let's verify that the mass matrix is correctly applied in the sampler. We'll create a custom mass matrix and check how it affects the momentum sampling and kinetic energy computation.

In [ ]:
# Define a custom mass matrix
inverse_mass_matrix = jnp.array([[2.0, 0.0], [0.0, 0.5]])

# Create a metric with this mass matrix
metric = default_metric(inverse_mass_matrix)


# Test momentum sampling
def test_momentum_sampling():
    position = jnp.array([1.0, 2.0])
    key1, key2 = random.split(key)

    # Sample momentum using the metric
    momentum1 = metric.sample_momentum(key1, position)
    momentum2 = metric.sample_momentum(key2, position)

    print(f"Position: {position}")
    print(f"Sampled momentum 1: {momentum1}")
    print(f"Sampled momentum 2: {momentum2}")

    # Compute kinetic energy
    ke1 = metric.kinetic_energy(momentum1)
    ke2 = metric.kinetic_energy(momentum2)

    print(f"Kinetic energy 1: {ke1}")
    print(f"Kinetic energy 2: {ke2}")

    # Verify that the kinetic energy is correctly computed
    # For a diagonal mass matrix M = diag(m1, m2), the kinetic energy is
    # K(p) = 0.5 * (p1^2/m1 + p2^2/m2)
    # For our inverse mass matrix M^(-1) = diag(2.0, 0.5), the kinetic energy is
    # K(p) = 0.5 * (2.0*p1^2 + 0.5*p2^2)

    # Manual calculation
    manual_ke1 = 0.5 * (2.0 * momentum1[0] ** 2 + 0.5 * momentum1[1] ** 2)
    manual_ke2 = 0.5 * (2.0 * momentum2[0] ** 2 + 0.5 * momentum2[1] ** 2)

    print(f"Manual kinetic energy 1: {manual_ke1}")
    print(f"Manual kinetic energy 2: {manual_ke2}")

    # Check if they match
    print(
        f"Kinetic energy matches manual calculation: {jnp.allclose(ke1, manual_ke1) and jnp.allclose(ke2, manual_ke2)}"
    )


test_momentum_sampling()

## 3. Test Energy Change Computation

Now let's verify that the energy change is correctly computed during the sampling process.

In [ ]:
def test_energy_change():
    # Initialize the sampler
    position = jnp.array([1.0, 2.0])
    L = 1.0  # Langevin parameter
    step_size = 0.1  # Step size

    # Create the sampler
    sampler = as_top_level_api(
        logdensity_fn=logdensity_fn,
        L=L,
        step_size=step_size,
        integrator=velocity_verlet,
        inverse_mass_matrix=inverse_mass_matrix,
    )

    # Initialize the state
    key1, key2 = random.split(key)
    state = sampler.init(position, key1)

    print(f"Initial position: {state.position}")
    print(f"Initial momentum: {state.momentum}")
    print(f"Initial logdensity: {state.logdensity}")

    # Compute initial energy
    initial_kinetic = metric.kinetic_energy(state.momentum)
    initial_potential = -state.logdensity
    initial_total = initial_kinetic + initial_potential

    print(f"Initial kinetic energy: {initial_kinetic}")
    print(f"Initial potential energy: {initial_potential}")
    print(f"Initial total energy: {initial_total}")

    # Take a step
    new_state, info = sampler.step(key2, state)

    print(f"\nNew position: {new_state.position}")
    print(f"New momentum: {new_state.momentum}")
    print(f"New logdensity: {new_state.logdensity}")

    # Compute new energy
    new_kinetic = metric.kinetic_energy(new_state.momentum)
    new_potential = -new_state.logdensity
    new_total = new_kinetic + new_potential

    print(f"New kinetic energy: {new_kinetic}")
    print(f"New potential energy: {new_potential}")
    print(f"New total energy: {new_total}")

    # Compute energy change manually
    manual_energy_change = new_total - initial_total

    print(f"\nEnergy change from info: {info.energy_change}")
    print(f"Energy change manual: {manual_energy_change}")
    print(f"Kinetic change from info: {info.kinetic_change}")
    print(f"Kinetic change manual: {new_kinetic - initial_kinetic}")

    # Check if they match
    print(
        f"Energy change matches manual calculation: {jnp.allclose(info.energy_change, manual_energy_change)}"
    )
    print(
        f"Kinetic change matches manual calculation: {jnp.allclose(info.kinetic_change, new_kinetic - initial_kinetic)}"
    )


test_energy_change()

## 4. Run the Sampler and Visualize Results

Now let's run the sampler for multiple steps and visualize the results.

In [ ]:
def run_sampler(num_steps=1000):
    # Initialize the sampler
    position = jnp.array([1.0, 2.0])
    L = 1.0  # Langevin parameter
    step_size = 0.1  # Step size

    # Create the sampler
    sampler = as_top_level_api(
        logdensity_fn=logdensity_fn,
        L=L,
        step_size=step_size,
        integrator=velocity_verlet,
        inverse_mass_matrix=inverse_mass_matrix,
    )

    # Initialize the state
    key1, key2 = random.split(key)
    state = sampler.init(position, key1)

    # Run the sampler
    positions = [state.position]
    energies = []

    for i in range(num_steps):
        key2, key3 = random.split(key2)
        state, info = sampler.step(key2, state)
        positions.append(state.position)
        energies.append(info.energy_change)

    return jnp.array(positions), jnp.array(energies)


positions, energies = run_sampler()


# Plot the trajectory
def plot_trajectory(positions):
    plt.figure(figsize=(10, 8))

    # Plot the target distribution
    x = np.linspace(-4, 4, 100)
    y = np.linspace(-4, 4, 100)
    X, Y = np.meshgrid(x, y)

    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = np.exp(logdensity_fn(jnp.array([X[i, j], Y[i, j]])))

    plt.contourf(X, Y, Z, levels=20, cmap="viridis", alpha=0.3)
    plt.colorbar(label="Density")

    # Plot the trajectory
    plt.plot(positions[:, 0], positions[:, 1], "r-", alpha=0.7, label="Trajectory")
    plt.scatter(positions[0, 0], positions[0, 1], color="green", s=100, label="Start")
    plt.scatter(positions[-1, 0], positions[-1, 1], color="blue", s=100, label="End")

    plt.title("Sampler Trajectory")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.legend()
    plt.show()


plot_trajectory(positions)

# Plot energy changes
plt.figure(figsize=(10, 6))
plt.plot(energies)
plt.title("Energy Changes")
plt.xlabel("Step")
plt.ylabel("Energy Change")
plt.grid(True)
plt.show()

# Plot histogram of positions
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(positions[:, 0], bins=30, density=True, alpha=0.7)
plt.title("Histogram of X")
plt.xlabel("X")
plt.ylabel("Density")

plt.subplot(1, 2, 2)
plt.hist(positions[:, 1], bins=30, density=True, alpha=0.7)
plt.title("Histogram of Y")
plt.xlabel("Y")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

## 5. Test with Different Mass Matrices

Let's test the sampler with different mass matrices to see how they affect the sampling behavior.

In [ ]:
def test_different_mass_matrices():
    # Define different mass matrices
    mass_matrices = [
        jnp.array([[1.0, 0.0], [0.0, 1.0]]),  # Identity
        jnp.array([[2.0, 0.0], [0.0, 0.5]]),  # Diagonal with different scales
        jnp.array([[1.0, 0.5], [0.5, 1.0]]),  # Correlated
    ]

    # Run the sampler with each mass matrix
    for i, inverse_mass_matrix in enumerate(mass_matrices):
        print(f"\nMass Matrix {i+1}:\n{inverse_mass_matrix}")

        # Create the sampler
        sampler = as_top_level_api(
            logdensity_fn=logdensity_fn,
            L=1.0,
            step_size=0.1,
            integrator=velocity_verlet,
            inverse_mass_matrix=inverse_mass_matrix,
        )

        # Initialize the state
        key1, key2 = random.split(key)
        state = sampler.init(jnp.array([1.0, 2.0]), key1)

        # Run the sampler
        positions = [state.position]

        for _ in range(100):
            key2, key3 = random.split(key2)
            state, _ = sampler.step(key2, state)
            positions.append(state.position)

        positions = jnp.array(positions)

        # Plot the trajectory
        plt.figure(figsize=(10, 8))

        # Plot the target distribution
        x = np.linspace(-4, 4, 100)
        y = np.linspace(-4, 4, 100)
        X, Y = np.meshgrid(x, y)

        Z = np.zeros_like(X)
        for i in range(X.shape[0]):
            for j in range(X.shape[1]):
                Z[i, j] = np.exp(logdensity_fn(jnp.array([X[i, j], Y[i, j]])))

        plt.contourf(X, Y, Z, levels=20, cmap="viridis", alpha=0.3)
        plt.colorbar(label="Density")

        # Plot the trajectory
        plt.plot(positions[:, 0], positions[:, 1], "r-", alpha=0.7, label="Trajectory")
        plt.scatter(
            positions[0, 0], positions[0, 1], color="green", s=100, label="Start"
        )
        plt.scatter(
            positions[-1, 0], positions[-1, 1], color="blue", s=100, label="End"
        )

        plt.title(f"Sampler Trajectory with Mass Matrix {i+1}")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.legend()
        plt.show()


test_different_mass_matrices()

## 6. Conclusion

In this notebook, we've tested the underdamped Langevin sampler in Blackjax, focusing on:

1. Verifying the mass matrix application in momentum sampling and kinetic energy computation
2. Confirming the energy change computation during the sampling process
3. Visualizing the sampler's behavior with different mass matrices

The tests show that:
- The mass matrix is correctly applied in momentum sampling and kinetic energy computation
- The energy change is correctly computed during the sampling process
- Different mass matrices affect the sampling behavior in expected ways

These results confirm that the underdamped Langevin sampler in Blackjax is working correctly with respect to mass matrix application and energy change computation.